# Macro Density Analysis: Nutritional Structure Exploration

This notebook explores the latent structure of food nutrition using multivariate statistical analysis and unsupervised learning techniques.

## Research Objective

This project investigates the following question:
- Can foods be grouped by nutritional density patterns?
- What latent dimensions define the structure of food composition?
- Can multivariate statistical techniques reveal hidden nutritional clusters?

Rather than analyzing nutruents independently, this study treats nutrition as a high-dimensional structured space.

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
sns.set(style="whitegrid")

## Dataset Overview

The dataset used in this analysis is the USDA Food Composition Dataset, which contains detailed nutritional information for a wide range of foods.
Each food item includes macro and micronutrient measurements, allowing for statistical exploration of nutritional patterns.

In [2]:
df = pd.read_csv("../data/USDA.csv")
df.head()

,ID,Description,Calories,Protein,TotalFat,Carbohydrate,Sodium,SaturatedFat,Cholesterol,Sugar,Calcium,Iron,Potassium,VitaminC,VitaminE,VitaminD
0,1001,"BUTTER,WITH SALT",717.0,0.85,81.11,0.06,714.0,51.368,215.0,0.06,24.0,0.02,24.0,0.0,2.32,1.5
1,1002,"BUTTER,WHIPPED,WITH SALT",717.0,0.85,81.11,0.06,827.0,50.489,219.0,0.06,24.0,0.16,26.0,0.0,2.32,1.5
2,1003,"BUTTER OIL,ANHYDROUS",876.0,0.28,99.48,0.00,2.0,61.924,256.0,0.00,4.0,0.00,5.0,0.0,2.80,1.8
3,1004,"CHEESE,BLUE",353.0,21.40,28.74,2.34,1395.0,18.669,75.0,0.50,528.0,0.31,256.0,0.0,0.25,0.5
4,1005,"CHEESE,BRICK",371.0,23.24,29.68,2.79,560.0,18.764,94.0,0.51,674.0,0.43,136.0,0.0,0.26,0.5


## Dataset Structure

Before conducting statistical analysis, we first examine the structure of the dataset, including the number of observations, feature types, and potential missing values.

In [3]:
df.shape

(7058, 16)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7058 entries, 0 to 7057
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ID            7058 non-null   int64  
 1   Description   7058 non-null   str    
 2   Calories      7057 non-null   float64
 3   Protein       7057 non-null   float64
 4   TotalFat      7057 non-null   float64
 5   Carbohydrate  7057 non-null   float64
 6   Sodium        6974 non-null   float64
 7   SaturatedFat  6757 non-null   float64
 8   Cholesterol   6770 non-null   float64
 9   Sugar         5148 non-null   float64
 10  Calcium       6922 non-null   float64
 11  Iron          6935 non-null   float64
 12  Potassium     6649 non-null   float64
 13  VitaminC      6726 non-null   float64
 14  VitaminE      4338 non-null   float64
 15  VitaminD      4224 non-null   float64
dtypes: float64(14), int64(1), str(1)
memory usage: 882.4 KB


In [5]:
df.describe()

,ID,Calories,Protein,TotalFat,Carbohydrate,Sodium,SaturatedFat,Cholesterol,Sugar,Calcium,Iron,Potassium,VitaminC,VitaminE,VitaminD
count,7058.000000,7057.000000,7057.000000,7057.000000,7057.000000,6974.000000,6757.000000,6770.000000,5148.000000,6922.000000,6935.000000,6649.000000,6726.000000,4338.000000,4224.000000
mean,14259.821196,219.695338,11.710368,10.320614,20.697860,322.059220,3.452267,41.551994,8.256540,73.530627,2.828368,301.357949,9.435980,1.487462,0.576918
std,8577.179705,172.198755,10.919356,16.814191,27.630443,1045.416931,6.921267,122.963028,15.361509,222.445338,6.019878,415.638949,71.256536,5.386914,4.301147
min,1001.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,8387.250000,85.000000,2.290000,0.720000,0.000000,37.000000,0.172000,0.000000,0.000000,9.000000,0.520000,135.000000,0.000000,0.120000,0.000000
50%,13293.500000,181.000000,8.200000,4.370000,7.130000,79.000000,1.256000,3.000000,1.395000,19.000000,1.330000,250.000000,0.000000,0.270000,0.000000
75%,18336.750000,331.000000,20.430000,12.700000,28.170000,386.000000,4.028000,69.000000,7.875000,56.000000,2.620000,348.000000,3.100000,0.710000,0.100000
max,93600.000000,902.000000,88.320000,100.000000,100.000000,38758.000000,95.600000,3100.000000,99.800000,7364.000000,123.600000,16500.000000,2400.000000,149.400000,250.000000


In [6]:
# Missing values check
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

VitaminD        2834
VitaminE        2720
Sugar           1910
Potassium        409
VitaminC         332
SaturatedFat     301
Cholesterol      288
Calcium          136
Iron             123
Sodium            84
Calories           1
Protein            1
TotalFat           1
Carbohydrate       1
dtype: int64

### Missing Ratio Analysis
To determine whether missing values require treatment, we calculate the proportion of missing values per feature.

In [7]:
missing_ratio = df.isnull().mean().sort_values(ascending=False)
missing_ratio

VitaminD        0.401530
VitaminE        0.385378
Sugar           0.270615
Potassium       0.057948
VitaminC        0.047039
SaturatedFat    0.042647
Cholesterol     0.040805
Calcium         0.019269
Iron            0.017427
Sodium          0.011901
Calories        0.000142
Protein         0.000142
TotalFat        0.000142
Carbohydrate    0.000142
Description     0.000000
ID              0.000000
dtype: float64

In [8]:
# Create filled version(do not overwrite original)
df_filled = df.fillna(0)

# Confirm
df_filled.isnull().sum().sum()

np.int64(0)

In [9]:
df.dtypes
# Check non-numeric columns (except indentifiers / names if any)
non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
non_numeric_cols

['Description']

In [10]:
# Percentile-based extreme value scan
summary = df_filled.describe(percentiles=[0.01, 0.05, 0.95, 0.99]).T
summary["median"] = df_filled.median(numeric_only=True)
summary[["min", "1%", "5%", "median", "95%", "99%", "max"]]

,min,1%,5%,median,95%,99%,max
ID,1001.0,1079.57,4541.85,13293.500,35022.1500,43334.87000,93600.00
Calories,0.0,7.57,24.00,181.000,527.0000,884.00000,902.00
Protein,0.0,0.00,0.07,8.200,29.2300,36.16430,88.32
TotalFat,0.0,0.00,0.05,4.365,37.1130,100.00000,100.00
Carbohydrate,0.0,0.00,0.00,7.130,80.0000,89.44300,100.00
Sodium,0.0,0.00,1.00,77.000,1089.0000,2071.86000,38758.00
SaturatedFat,0.0,0.00,0.00,1.120,12.6400,28.45611,95.60
Cholesterol,0.0,0.00,0.00,2.000,106.1500,360.72000,3100.00
Sugar,0.0,0.00,0.00,0.000,37.8100,65.21500,99.80
Calcium,0.0,0.00,0.00,19.000,279.1500,960.43000,7364.00


In [11]:
# Apply log1p transformation to numeric features(excluding ID)
numeric_cols = [col for col in df_filled.columns if col not in ["ID", "Description"]]
df_log = df_filled.copy()
df_log[numeric_cols] = np.log1p(df_log[numeric_cols])

# Reconstruct the modeling space
X = df_log[numeric_cols].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Scaled matric shape:", X_scaled.shape)

Scaled matric shape: (7058, 14)


## Nutrient Distribution Analysis

To understand the statistical structure of the dataset, we examine the distribution of several key nutritional variables, including calories, protein, fat, and carbohydrates.
Distribution plots help reveal skewness, outliers, and general patterns in nutritional density.

In [12]:
df_filled.describe().T

,count,mean,std,min,25%,50%,75%,max
ID,7058.0,14259.821196,8577.179705,1001.0,8387.250,13293.500,18336.75000,93600.00
Calories,7058.0,219.664211,172.206410,0.0,85.000,181.000,331.00000,902.00
Protein,7058.0,11.708709,10.919472,0.0,2.290,8.200,20.42750,88.32
TotalFat,7058.0,10.319151,16.813449,0.0,0.720,4.365,12.69750,100.00
Carbohydrate,7058.0,20.694928,27.629584,0.0,0.000,7.130,28.17000,100.00
Sodium,7058.0,318.226268,1039.763266,0.0,35.000,77.000,383.00000,38758.00
SaturatedFat,7058.0,3.305040,6.807890,0.0,0.114,1.120,3.81225,95.60
Cholesterol,7058.0,39.856475,120.708085,0.0,0.000,2.000,68.00000,3100.00
Sugar,7058.0,6.022198,13.622256,0.0,0.000,0.000,4.07750,99.80
Calcium,7058.0,72.113772,220.523285,0.0,9.000,19.000,54.00000,7364.00


### Distribution Summary Interpretation

- The majority of nutritional variables demonstrate right-skewed behavior, as indicated by mean values exceeding median values.
- Micronutrients such as Vitamin D and Vitamin C are highly sparse, with median values equal to zero, indicating uneven distribution across foods.
- Sodium, calcium and Potassium show extreme heavy-tail properties, suggesting potential influence of outliers in multivariate modeling.

In [13]:
# Skewness ranking
skewness = df_filled.skew(numeric_only=True).sort_values(ascending=False)
skewness

VitaminD        60.133350
VitaminC        27.115162
Sodium          20.674359
Cholesterol     14.999503
Potassium       14.365420
Calcium         13.284214
VitaminE        12.602929
SaturatedFat     6.881767
Iron             6.645845
TotalFat         3.331222
Sugar            3.199996
ID               1.681349
Calories         1.316567
Carbohydrate     1.275261
Protein          1.166856
dtype: float64

### Skewness structure interpertation

- The skewness ranking reveals a highly non-normal nutritional space.
- Several micronutrients(Vitamin D, VitaminC, Sodium, Cholesterol, Potassium, Calcium, VitaminE) exhibit extreme right-skewness (skewness > 10), indicating heavy-tailed and sparse distributions. These nutrients are present in very small amount in most foods but appear in extreme concentrations in specific food categories.
- Macronutrients such as Calories, Carbohydrate and Protein show moderate right-skew (skewness ≈ 1–1.5), suggesting more stable density patterns across food items.
- This confirms that the nutritional feature space is highly heterogeneous and may require transformation before multivariate modeling.

### Calorie distribution characteristics

- Calories exhibit moderate right-skew (skewness ≈ 1.3), indication that most foods cluster around lower to moderate energy density, while a smaller subset of high-calorie foods (e.g., oils and processed items) extends the upper tail.
- This suggests that energy density is relatively stable compared to micronutrients, yet still influenced by concentrated fat-based food categories.

### Protein Distribution Characteristics

- Protein exhibit moderate right-skew (skewness ≈ 1.7), suggesting that most foods contain lower to moderate protein density, while a smaller subset of high-protein foods (e.g., fishes, meats, legumes) extends the upper tail.
- Compared to micronutrients, protein distribution is less extreme and shows a more continuous density structure across food items.
This suggests that protein behaves as a core macronutrient axis rather than a sparse, burst type variable.

In [14]:
# Log1p transformation comparisons
numeric_cols = [c for c in df_filled.columns if c not in ["ID", "Description"]]

# Before(original skewness)
skew_before = df_filled[numeric_cols].skew(numeric_only=True).sort_values(ascending=False)

# Apply log1p transformation only to numeric columns
df_log = df_filled.copy()
df_log[numeric_cols] = np.log1p(df_log[numeric_cols])

#After (log1p skewness)
skew_after = df_log[numeric_cols].skew(numeric_only=True).sort_values(ascending=False)

#Comparison table
skew_compare = (
    pd.concat([skew_before.rename("before"), skew_after.rename("after")], axis=1)
      .assign(reduction=lambda d: d["before"] - d["after"])
      .sort_values("before", ascending=False)
)
skew_compare

,before,after,reduction
VitaminD,60.133350,4.460430,55.672919
VitaminC,27.115162,1.595856,25.519306
Sodium,20.674359,-0.480490,21.154849
Cholesterol,14.999503,0.361296,14.638207
Potassium,14.365420,-1.849521,16.214941
Calcium,13.284214,0.177845,13.106369
VitaminE,12.602929,3.430797,9.172132
SaturatedFat,6.881767,0.846549,6.035218
Iron,6.645845,1.357332,5.288513
TotalFat,3.331222,0.303905,3.027317


### Log-transformation effect on nutritional skewness

- Applying log1p transformation significantly reduces extreme right-skewness in micronutrients.
- Variables such as Vitamin D, Vitamin C, Sodium, and Calcium show dramatic skewness reduction, indicating that heavy-tail effects were primarily driven by extreme high-density outliers.

After transformation:
- Most variables fall within a moderate skewness range (-2 to 2).
- Several previously pathological distributions become approximately symmetric.
- A few variables exhibit mild left-skew due to compression of extreme values.

This confirms that log1p transformation stabilizes the nutritional feature space and is appropriate prior to multivariate modeling (PCA and clustering).